### Training Overview

This notebook fine-tunes Gemma 4 E2B using Unsloth and LoRA/PEFT for CUAD-style legal clause extraction. The training pipeline prepares the CUAD dataset, applies answer-aware chunking to keep relevant clauses within the model context, formats examples using the Gemma chat template, and performs supervised fine-tuning. Checkpointing and validation are included to support long-running training sessions and recovery. The final model is evaluated using precision, recall, F1, exact match, no-answer accuracy, macro-F1, and evidence grounding rate.

In [1]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 22.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 49.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━

### Library Imports and Dependencies

This cell imports the libraries required for the complete CUAD fine-tuning and evaluation workflow. It includes Unsloth and PyTorch for model loading and training, Hugging Face `datasets`, Transformers, and TRL for dataset handling and supervised fine-tuning, along with supporting libraries for data processing, evaluation, checkpoint management, and Kaggle secret access.

In [2]:
import unsloth
import torch
import re
import string
import collections
import numpy as np
import json
import inspect
import trl
import json
import os
import glob
import functools
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from sklearn.metrics import average_precision_score
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split
from kaggle_secrets import UserSecretsClient
from transformers.trainer_utils import get_last_checkpoint

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Hugging Face Authentication

This cell securely retrieves the Hugging Face access token from Kaggle Secrets. The token is used to authenticate access to gated Hugging Face models and resources without exposing the token directly in the notebook.

In [3]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF-Token")

### Model Loading and Configuration

This cell loads the Gemma 4 E2B instruction-tuned model using Unsloth's `FastLanguageModel`. The model uses automatic dtype detection, a 2048-token sequence length, and 4-bit quantization to reduce GPU memory usage during fine-tuning. Full model fine-tuning is disabled so that the training can use parameter-efficient LoRA/PEFT adapters. The Hugging Face token is supplied for authenticated model access.

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it", # Change this to unsloth/gemma-4-E2B-it etc
    dtype = None, # None for auto detection
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = secret_value_0, # HF Token for gated models
)


==((====))==  Unsloth 2026.8.19: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [5]:
# # 1. Load Gemma 2 9B (4-bit quantized)
# model_name = "unsloth/gemma-2-9b-it-bnb-4bit"

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=model_name,
#     max_seq_length=4096,
#     dtype=None,
#     load_in_4bit=True,
# )

### LoRA Adapter Configuration

This cell adds LoRA adapters to the Gemma 4 E2B model for parameter-efficient fine-tuning. A rank of 16 is used across the attention and feed-forward projection layers, allowing the model to learn the CUAD-specific extraction task while updating only a small fraction of the model's parameters. Unsloth gradient checkpointing is enabled to reduce GPU memory usage during training.

In [6]:
# 2. Add LoRA Adapters (Rank 16 as recommended)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


# 2. DATA PREPARATION

## Data Preparation — CUAD v1

This section prepares the official **CUAD v1** dataset for supervised fine-tuning while preserving the original contract-level structure of the dataset.

### 1. Dataset Acquisition
The official CUAD v1 SQuAD-style JSON is downloaded directly from the Atticus Project's Hugging Face dataset repository. The JSON contains contracts, contract sections, questions corresponding to CUAD categories, and their annotated answer spans.

### 2. Conversion to a Hugging Face Dataset
The nested CUAD JSON structure is flattened into individual examples containing:

- **ID** — unique question/example identifier
- **Question** — CUAD legal category and extraction instruction
- **Context** — contract section containing the relevant evidence
- **Answers** — annotated answer spans
- **Contract** — contract title used for contract-level splitting

This produces one dataset entry for each contract-section/category combination.

### 3. Contract-Level Train/Validation/Test Split
To prevent data leakage, the split is performed **by contract rather than by individual examples**. Contracts are divided into:

- **80% Training**
- **10% Validation**
- **10% Testing**

All examples belonging to the same contract remain within the same split. This is important because a single CUAD contract can contain many different category questions; randomly splitting individual examples could otherwise expose the model to portions of the same contract during both training and testing.

### 4. Split Integrity Verification
After filtering the examples according to their assigned contracts, the intersections between the three contract sets are checked:

```text
Train ∩ Validation = 0
Train ∩ Test       = 0
Validation ∩ Test   = 0

In [7]:
# 3. Downloading the official CUAD v1 SQuAD-style JSON

cuad_json_path = hf_hub_download(
    repo_id="theatticusproject/cuad",
    filename="CUAD_v1/CUAD_v1.json",
    repo_type="dataset"
)

print("CUAD file:", cuad_json_path)

CUAD_v1/CUAD_v1.json:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

CUAD file: /root/.cache/huggingface/hub/datasets--theatticusproject--cuad/snapshots/a3c393f5d103fd0c516374e4fdff676c8176dcb1/CUAD_v1/CUAD_v1.json


In [8]:
with open(cuad_json_path, "r", encoding="utf-8") as f:
    cuad_data = json.load(f)

print(cuad_data.keys())

dict_keys(['version', 'data'])


In [9]:
examples = []

for contract in cuad_data["data"]:
    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]

        for qa in paragraph["qas"]:
            examples.append({
                "id": qa["id"],
                "question": qa["question"],
                "context": context,
                "answers": qa["answers"],
                "contract": contract.get("title", "")
            })

print("Total examples:", len(examples))

Total examples: 20910


In [10]:
dataset = Dataset.from_list(examples)

print(dataset)
print(dataset.column_names)

Dataset({
    features: ['id', 'question', 'context', 'answers', 'contract'],
    num_rows: 20910
})
['id', 'question', 'context', 'answers', 'contract']


In [11]:
# Get unique contract IDs
contracts = list(set(dataset["contract"]))

print("Total contracts:", len(contracts))

# First split: 80% train, 20% temporary
train_contracts, temp_contracts = train_test_split(
    contracts,
    test_size=0.20,
    random_state=3407
)

# Second split: 10% validation, 10% test
val_contracts, test_contracts = train_test_split(
    temp_contracts,
    test_size=0.50,
    random_state=3407
)

print("\nContract split:")
print("Train:", len(train_contracts))
print("Validation:", len(val_contracts))
print("Test:", len(test_contracts))

Total contracts: 510

Contract split:
Train: 408
Validation: 51
Test: 51


In [12]:
train_contracts = set(train_contracts)
val_contracts = set(val_contracts)
test_contracts = set(test_contracts)

train_dataset_raw = dataset.filter(
    lambda x: x["contract"] in train_contracts
)

val_dataset_raw = dataset.filter(
    lambda x: x["contract"] in val_contracts
)

test_dataset_raw = dataset.filter(
    lambda x: x["contract"] in test_contracts
)

print("Examples:")
print("Train:", len(train_dataset_raw))
print("Validation:", len(val_dataset_raw))
print("Test:", len(test_dataset_raw))

Filter:   0%|          | 0/20910 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20910 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20910 [00:00<?, ? examples/s]

Examples:
Train: 16728
Validation: 2091
Test: 2091


In [13]:
train_ids = set(train_dataset_raw["contract"])
val_ids = set(val_dataset_raw["contract"])
test_ids = set(test_dataset_raw["contract"])

print("Train ∩ Validation:", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(val_ids & test_ids))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [14]:
train_categories = set(train_dataset_raw["question"])
val_categories = set(val_dataset_raw["question"])
test_categories = set(test_dataset_raw["question"])

print("Train categories:", len(train_categories))
print("Validation categories:", len(val_categories))
print("Test categories:", len(test_categories))

Train categories: 41
Validation categories: 41
Test categories: 41


## Answer-Aware Chunking and Prompt Construction

This section prepares each CUAD example for efficient supervised fine-tuning by combining a structured extraction prompt with **answer-aware context chunking**.

### 1. Standardized Extraction Task

The `render_user_message()` function defines a consistent instruction format for the model. Each example provides:

- The CUAD category being evaluated
- A relevant contract section
- A strict extraction instruction

For positive examples, the model is instructed to return **only the exact text span** corresponding to the category. For negative examples, it must return the standardized `NO_ANSWER` response. This prevents explanations, summaries, or hallucinated content from becoming part of the target output.

### 2. Context Token Budget

A context budget of **1,300 tokens** is used for the contract section. This leaves additional room within the model's 2,048-token sequence limit for the task instructions, category, chat formatting, and generated response.

### 3. Cached Tokenization

CUAD contains multiple category questions for the same contract. Since the same contract context can therefore be processed repeatedly, tokenization with character-offset mapping is cached using `functools.lru_cache`.

The character offsets are important because CUAD provides the answer position in the original contract text. They allow the annotated answer to be mapped from character positions to token positions efficiently.

### 4. Answer-Aware Chunking

The `make_answer_aware_chunk()` function creates a compact contract section for each training example.

For **positive examples**, the annotated answer is located within the original contract and the 1,300-token window is positioned around it. The function also verifies that the complete answer remains inside the resulting chunk.

For **no-answer examples**, the first available portion of the contract is used and the target is set to `NO_ANSWER`.

### 5. Safety and Integrity Checks

Several checks are performed during preprocessing to prevent malformed training examples:

- The annotated answer must be found within the original context.
- The answer must fit within the configured token budget.
- The complete answer must remain inside the generated chunk.
- Token-to-character offsets must successfully map the CUAD annotation.

This preprocessing significantly reduces the amount of unnecessary contract text presented to the model while ensuring that positive training examples retain the evidence required to learn the extraction task.

In [15]:
NO_ANSWER = "NO_ANSWER"

def render_user_message(category, context_text):
    return f"""You are a legal contract extraction assistant.

Task:
Determine whether the contract contains text relevant to the following CUAD category.

Category:
{category}

Contract Section:
{context_text}

Instructions:
- If the category is present, return only the exact text span from the contract.
- If the category is not present, return exactly: {NO_ANSWER}
- Do not explain your answer.
- Do not summarize.
- Do not add text that is not present in the contract."""

In [16]:
CONTEXT_TOKEN_BUDGET = 1300


def get_base_tokenizer(processor):
    """
    Gemma 4 exposes a processor rather than a plain tokenizer.
    We use its underlying fast tokenizer when available.
    """
    if hasattr(processor, "tokenizer"):
        return processor.tokenizer

    return processor


base_tokenizer = get_base_tokenizer(tokenizer)

"""
Cache tokenization per unique contract context.

CUAD stores ONE contract context shared by ~41 category
questions (20,910 examples / 510 contracts ≈ 41). Without
caching, the same multi-thousand-token contract text gets
re-tokenized with offsets up to 41 times. This cache turns
that into a single tokenization per contract.
"""




@functools.lru_cache(maxsize=600)
def _tokenize_with_offsets_cached(context):
    encoded = base_tokenizer(
        context,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )
    return tuple(encoded["input_ids"]), tuple(encoded["offset_mapping"])


def tokenize_with_offsets(text):
    """
    Tokenize text while retaining character offsets.

    This lets us map the CUAD answer's character position
    back to its token position. Cached per unique contract text.
    """
    input_ids, offsets = _tokenize_with_offsets_cached(text)
    return {"input_ids": list(input_ids), "offset_mapping": list(offsets)}


def make_answer_aware_chunk(example):
    """
    Create a context chunk that keeps the CUAD answer inside
    the model's context window.

    Positive example:
        center the chunk around the answer.

    No-answer example:
        use the first context window.
    """

    context = example["context"]

     #Determine whether this example has an answer

    if len(example["answers"]) > 0:

        answer = example["answers"][0]["text"]

        # CUAD normally provides the answer_start.
        answer_start = example["answers"][0].get("answer_start", None)

        # If answer_start is unavailable, find the answer manually.
        if answer_start is None:
            answer_start = context.find(answer)

        if answer_start == -1:
            raise ValueError(
                "CUAD answer could not be located inside context."
            )

        answer_end = answer_start + len(answer)

    else:

        answer = NO_ANSWER
        answer_start = None
        answer_end = None


    # Tokenize context with character offsets

    encoded = tokenize_with_offsets(context)

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    total_tokens = len(input_ids)

    # NO-ANSWER example

    if answer_start is None:

        end_token = min(
            CONTEXT_TOKEN_BUDGET,
            total_tokens
        )

        chunk_start_token = 0
        chunk_end_token = end_token

        chunk = base_tokenizer.decode(
            input_ids[chunk_start_token:chunk_end_token],
            skip_special_tokens=True
        )

        return {
            "chunk_context": chunk,
            "answer": NO_ANSWER,
            "has_answer": False,
        }


    # POSITIVE example
    # Find token range covering the answer

    answer_start_token = None
    answer_end_token = None

    for i, (start, end) in enumerate(offsets):

        # Token overlaps answer start
        if start <= answer_start < end:
            answer_start_token = i

        # Token overlaps answer end
        if start < answer_end <= end:
            answer_end_token = i + 1
            break

    if answer_start_token is None:
        raise ValueError(
            "Could not map answer start to token position."
        )

    if answer_end_token is None:
        answer_end_token = answer_start_token + 1


    answer_token_length = (
        answer_end_token - answer_start_token
    )


    # Make sure the answer itself fits

    if answer_token_length > CONTEXT_TOKEN_BUDGET:

        raise ValueError(
            f"Answer itself is {answer_token_length} tokens, "
            f"which exceeds the {CONTEXT_TOKEN_BUDGET}-token "
            f"context budget."
        )


    # Center the chunk around the answer

    half_window = CONTEXT_TOKEN_BUDGET // 2

    chunk_start_token = max(
        0,
        answer_start_token - half_window
    )

    chunk_end_token = (
        chunk_start_token + CONTEXT_TOKEN_BUDGET
    )


    # Shift window backwards if it goes past document end
    if chunk_end_token > total_tokens:

        chunk_end_token = total_tokens

        chunk_start_token = max(
            0,
            chunk_end_token - CONTEXT_TOKEN_BUDGET
        )


    # Safety check: answer must be completely inside chunk

    if not (
        chunk_start_token <= answer_start_token
        and
        answer_end_token <= chunk_end_token
    ):

        raise ValueError(
            "Answer was not fully contained inside chunk."
        )

    # Decode chunk

    chunk = base_tokenizer.decode(
        input_ids[chunk_start_token:chunk_end_token],
        skip_special_tokens=True
    )


    return {
        "chunk_context": chunk,
        "answer": answer,
        "has_answer": True,
    }


In [17]:
"""
Quick sanity check on one positive example and one no-answer example,
so we catch problems here instead of an hour into training.
"""

positive_example = next(
    ex for ex in train_dataset_raw if len(ex["answers"]) > 0
)
no_answer_example = next(
    ex for ex in train_dataset_raw if len(ex["answers"]) == 0
)

for label, ex in [("Positive", positive_example), ("No-answer", no_answer_example)]:
    chunk = make_answer_aware_chunk(ex)
    print(f"--- {label} example ---")
    print("Has answer:", chunk["has_answer"])
    print("Answer:", chunk["answer"][:200])
    print("Chunk length (chars):", len(chunk["chunk_context"]))

    if chunk["has_answer"]:
        assert chunk["answer"] in chunk["chunk_context"], "Answer missing from its own chunk!"
        print("Answer confirmed inside chunk.")
    print()


--- Positive example ---
Has answer: True
Answer: DISTRIBUTOR AGREEMENT
Chunk length (chars): 6865
Answer confirmed inside chunk.

--- No-answer example ---
Has answer: False
Answer: NO_ANSWER
Chunk length (chars): 6865



## Build the Training Text (chunk + template in one pass)

Each raw example is chunked *and* rendered through the chat template in a single `.map()` call, so `train_dataset` / `val_dataset` / `test_dataset` are now built from `chunk_context` (≈1300 tokens) instead of the raw contract `context` (which averaged ~12.8k tokens and ran up to 63k).

In [18]:
NO_ANSWER = "NO_ANSWER"


def render_user_message(category, context_text):
    return f"""You are a legal contract extraction assistant.

Task:
Determine whether the contract contains text relevant to the following CUAD category.

Category:
{category}

Contract Section:
{context_text}

Instructions:
- If the category is present, return only the exact text span from the contract.
- If the category is not present, return exactly: {NO_ANSWER}
- Do not explain your answer.
- Do not summarize.
- Do not add text that is not present in the contract."""


def build_training_example(example):
    chunk = make_answer_aware_chunk(example)

    category = example["question"]
    user_content = render_user_message(category, chunk["chunk_context"])

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": chunk["answer"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text, "has_answer": chunk["has_answer"]}


train_dataset = train_dataset_raw.map(
    build_training_example,
    remove_columns=train_dataset_raw.column_names,
)

val_dataset = val_dataset_raw.map(
    build_training_example,
    remove_columns=val_dataset_raw.column_names,
)

test_dataset = test_dataset_raw.map(
    build_training_example,
    remove_columns=test_dataset_raw.column_names,
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


Map:   0%|          | 0/16728 [00:00<?, ? examples/s]

Map:   0%|          | 0/2091 [00:00<?, ? examples/s]

Map:   0%|          | 0/2091 [00:00<?, ? examples/s]

Train: 16728
Validation: 2091
Test: 2091


### Sanity-check sequence lengths *before* training

In [19]:
lengths = []

subset = train_dataset.select(
    range(min(2000, len(train_dataset)))
)

for example in subset:

    encoded = tokenizer(
        text=[example["text"]],
        add_special_tokens=True,
        truncation=False,
    )

    lengths.append(len(encoded["input_ids"][0]))

print("Examples analyzed:", len(lengths))
print("Minimum:", np.min(lengths))
print("Maximum:", np.max(lengths))
print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("90th percentile:", np.percentile(lengths, 90))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))

for limit in [1024, 1536, 2048]:
    exceeded = sum(x > limit for x in lengths)

    print(
        f"{limit} tokens → "
        f"{exceeded}/{len(lengths)} "
        f"({100 * exceeded / len(lengths):.2f}%) exceed limit"
    )

# Expectation: almost everything should now sit close to
# CONTEXT_TOKEN_BUDGET (1300) + a few hundred tokens of template/answer,
# comfortably under max_length=2048. If you still see examples near
# 2048+, shrink CONTEXT_TOKEN_BUDGET a bit further.


Examples analyzed: 2000
Minimum: 387
Maximum: 2006
Mean: 1433.48
Median: 1470.0
90th percentile: 1524.1000000000001
95th percentile: 1564.0
99th percentile: 1636.07
1024 tokens → 1918/2000 (95.90%) exceed limit
1536 tokens → 155/2000 (7.75%) exceed limit
2048 tokens → 0/2000 (0.00%) exceed limit


## Training Configuration and Checkpoint Management

This section defines the supervised fine-tuning process for the Gemma 4 E2B model using Unsloth's `SFTTrainer`. Because training is performed in a Kaggle environment and can require several hours, the training pipeline is designed to support efficient training, periodic validation, automatic checkpoint discovery, and recovery from interrupted sessions.

### Checkpoint Detection and Recovery

The first part of the section introduces safe checkpoint handling.

The `safe_get_last_checkpoint()` helper checks whether a directory exists before attempting to retrieve a Trainer checkpoint. This prevents errors when the output directory has not yet been created.

The notebook then searches both Kaggle storage locations:

- `/kaggle/working/` for checkpoints produced during the current session.
- `/kaggle/input/` for checkpoints attached from a previous Kaggle session.

The `find_resume_checkpoint()` function searches these locations for directories following the `checkpoint-N` naming convention and verifies that each candidate contains `trainer_state.json`. This prevents unrelated directories from being treated as valid training checkpoints.

When multiple checkpoints are found, the checkpoint with the highest step number is selected automatically.

The resulting behavior is:

No checkpoint found → start training from the beginning.

Checkpoint found → identify the latest checkpoint and resume training from that step.

This allows the same notebook to support both fresh training runs and interrupted/resumed runs without manually changing the training configuration.

### Training Configuration

The `SFTConfig` object defines the core supervised fine-tuning parameters.

**Batch size and gradient accumulation**

`per_device_train_batch_size=2` processes two examples per device at a time, while `gradient_accumulation_steps=4` accumulates gradients across four steps before updating the model.

This gives an effective batch size of approximately:

`2 × 4 = 8`

This approach keeps the memory requirement of each individual step manageable while still providing a larger effective batch during optimization.

**Training duration**

`num_train_epochs=1` performs one complete pass over the training dataset. The objective is to specialize the already capable Gemma instruction model for CUAD-style legal clause extraction rather than retrain its general language capabilities.

**Learning rate and warmup**

`learning_rate=2e-5` controls the magnitude of the LoRA parameter updates, while `warmup_steps=105` gradually introduces the learning rate during the initial part of training to improve optimization stability.

**Precision and optimizer**

The configuration automatically selects BF16 when the GPU supports it and otherwise falls back to FP16.

The optimizer is configured as `adamw_8bit`, reducing optimizer-state memory usage and making parameter-efficient fine-tuning more practical on the available Kaggle GPUs.

**Sequence length**

`max_length=2048` limits each training sequence to 2,048 tokens.

This works together with the earlier answer-aware chunking stage, where the contract context is limited to approximately 1,300 tokens. The remaining sequence capacity is therefore available for the category, extraction instructions, Gemma chat formatting, and expected response.

### Checkpointing Strategy

The trainer saves a checkpoint every 250 steps:

`save_strategy="steps"`  
`save_steps=250`  
`save_total_limit=3`

This creates regular recovery points during the long training process while limiting the number of retained checkpoints to control storage usage.

For example, the training directory may contain:

`checkpoint-250`  
`checkpoint-500`  
`checkpoint-750`

Older checkpoints can be removed automatically as new checkpoints are created according to `save_total_limit`.

Checkpointing is particularly important in Kaggle because training may be interrupted by session limits or restarts.

### Periodic Validation

Validation is performed every 250 training steps using:

`eval_strategy="steps"`  
`eval_steps=250`

To reduce the time spent repeatedly evaluating the model during training, the notebook uses a subset of up to 300 validation examples:

`eval_subset = val_dataset.select(range(min(300, len(val_dataset))))`

This provides a lightweight indication of validation loss during training without repeatedly evaluating the entire validation dataset.

A full validation or test evaluation can be performed separately after training for the final performance measurements.

### Best Checkpoint Selection

The trainer is configured to retain the model state associated with the lowest validation loss:

`load_best_model_at_end=True`  
`metric_for_best_model="eval_loss"`  
`greater_is_better=False`

This ensures that the checkpoint selection is based on validation performance rather than simply taking the final training step.

### Supervised Fine-Tuning

`SFTTrainer` receives the prepared `train_dataset` and the reduced `eval_subset`.

The training data already contains the Gemma-formatted `"text"` field generated during the prompt-construction stage. Each example therefore represents the complete supervised task:

Category + contract context → exact CUAD answer or `NO_ANSWER`.

During training, the trainer performs the forward pass, computes the supervised loss, updates the LoRA parameters, periodically evaluates the validation subset, and saves recovery checkpoints.

### Resumable Training

The selected checkpoint is passed directly to:

`trainer.train(resume_from_checkpoint=resume_checkpoint)`

If `resume_checkpoint` is `None`, training starts from the newly initialized LoRA model.

If a checkpoint is found, the Trainer restores the previous training state and continues from that step.

This makes the notebook suitable for repeated Kaggle sessions without requiring previously completed training steps to be repeated.

### Final Adapter Saving

After training completes, the trained adapter and tokenizer are saved to:

`/kaggle/working/adapter_1_gemma4_e2b_cuad`

Because the training uses LoRA/PEFT, the saved artifact represents the fine-tuned adapter rather than a complete copy of the base model. The adapter can later be loaded together with the original Gemma model for evaluation and inference.

### Overall Training Flow

Prepared CUAD dataset  
↓  
Search Kaggle for existing checkpoints  
↓  
Resume from the latest checkpoint, or start from scratch  
↓  
LoRA supervised fine-tuning  
↓  
Periodic validation every 250 steps  
↓  
Checkpoint saving every 250 steps  
↓  
Best checkpoint selected using validation loss  
↓  
Final LoRA adapter saved for evaluation and inference

This configuration balances parameter-efficient training, GPU memory usage, validation overhead, checkpoint recovery, and reliable long-running execution in the Kaggle environment.

In [20]:
print(inspect.signature(SFTTrainer))
print()
print("TRL version:", trl.__version__)

(model, args=None, data_collator=None, train_dataset=None, eval_dataset=None, processing_class=None, compute_loss_func=None, compute_metrics=None, callbacks=None, optimizer_cls_and_kwargs=None, preprocess_logits_for_metrics=None, peft_config=None, formatting_func=None, **kwargs)

TRL version: 0.24.0


In [21]:
def safe_get_last_checkpoint(folder):
    """get_last_checkpoint throws if the folder doesn't exist — guard it."""
    if os.path.isdir(folder):
        try:
            return get_last_checkpoint(folder)
        except Exception:
            return None
    return None

In [22]:
print("Kaggle inputs:")
for path in glob.glob("/kaggle/input/*"):
    print(" ", path)

print("\nCheckpoint search:")
for path in glob.glob("/kaggle/input/**/checkpoint-*", recursive=True):
    print(" ", path)

Kaggle inputs:
  /kaggle/input/datasets

Checkpoint search:
  /kaggle/input/datasets/glassuchan/checkpoints/checkpoint-1000


In [23]:
def find_resume_checkpoint(output_dir):
    """
    Find the latest Trainer checkpoint from either:

    1. /kaggle/working/<output_dir>
    2. Any attached dataset under /kaggle/input/

    Returns the checkpoint with the highest step number.
    """

    candidates = []

    # ---------------------------------------------------------
    # 1. Check the current Kaggle working directory
    # ---------------------------------------------------------

    working_output_dir = output_dir

    if not os.path.isabs(working_output_dir):
        working_output_dir = os.path.join(
            "/kaggle/working",
            working_output_dir
        )

    if os.path.isdir(working_output_dir):

        for path in glob.glob(
            os.path.join(working_output_dir, "checkpoint-*")
        ):

            if not os.path.isdir(path):
                continue

            if not re.match(r"checkpoint-\d+$", os.path.basename(path)):
                continue

            # Make sure this is actually a Trainer checkpoint
            if os.path.isfile(
                os.path.join(path, "trainer_state.json")
            ):
                candidates.append(path)
                print(f"[output]  found checkpoint: {path}")


    # ---------------------------------------------------------
    # 2. Search ALL attached Kaggle input datasets
    # ---------------------------------------------------------

    for path in glob.glob(
        "/kaggle/input/**/checkpoint-*",
        recursive=True
    ):

        if not os.path.isdir(path):
            continue

        # Only accept names like checkpoint-1000
        if not re.match(
            r"checkpoint-\d+$",
            os.path.basename(path)
        ):
            continue

        # Make sure this is actually a Trainer checkpoint
        if os.path.isfile(
            os.path.join(path, "trainer_state.json")
        ):
            candidates.append(path)
            print(f"[input]   found checkpoint: {path}")


    # ---------------------------------------------------------
    # 3. Nothing found
    # ---------------------------------------------------------

    if not candidates:

        print(
            "[resume] No valid checkpoint found. "
            "Training will start from scratch."
        )

        return None


    # ---------------------------------------------------------
    # 4. Select the checkpoint with the highest step
    # ---------------------------------------------------------

    latest_checkpoint = max(
        candidates,
        key=lambda p: int(
            re.search(
                r"checkpoint-(\d+)$",
                os.path.basename(p)
            ).group(1)
        )
    )

    step = re.search(
        r"checkpoint-(\d+)$",
        os.path.basename(latest_checkpoint)
    ).group(1)

    print(
        f"\n[resume] Latest checkpoint found:"
        f"\n         {latest_checkpoint}"
        f"\n         Step: {step}"
    )

    return latest_checkpoint

In [24]:
OUTPUT_DIR = "outputs_adapter1_gemma4_e2b"

resume_checkpoint = find_resume_checkpoint(OUTPUT_DIR)

if resume_checkpoint is not None:
    step = resume_checkpoint.rstrip("/").split("-")[-1]
    print(f"\nResuming training from step {step}: {resume_checkpoint}")
else:
    print("\nNo existing checkpoint found anywhere — starting training from scratch.")

[input]   found checkpoint: /kaggle/input/datasets/glassuchan/checkpoints/checkpoint-1000

[resume] Latest checkpoint found:
         /kaggle/input/datasets/glassuchan/checkpoints/checkpoint-1000
         Step: 1000

Resuming training from step 1000: /kaggle/input/datasets/glassuchan/checkpoints/checkpoint-1000


In [25]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    warmup_steps=105,
    learning_rate=2e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    seed=3407,
    report_to="none",
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=250,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=2048,
    dataset_num_proc=2,
)

# Periodic in-training eval on a smaller slice keeps the eval_steps=250 checkpoints
# fast — with the old unchunked data + full 2,091-example val set, every one of these
# was as expensive as a big chunk of training itself. Run a pass over the full
# val_dataset separately (or swap it back in here) for your real validation numbers.
eval_subset = val_dataset.select(range(min(300, len(val_dataset))))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_subset,
    dataset_text_field="text",
    args=training_args,
)

print("Starting Adapter 1 training — Gemma 4 E2B...")

trainer_stats = trainer.train(resume_from_checkpoint=resume_checkpoint)

adapter_path = "/kaggle/working/adapter_1_gemma4_e2b_cuad"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")


Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/16728 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Starting Adapter 1 training — Gemma 4 E2B...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,728 | Num Epochs = 1 | Total steps = 2,091
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
1250,1.969210,2.126935
1500,2.076322,2.211545
1750,2.011506,2.178496
2000,2.112789,2.201842
2091,2.021591,2.200219


Unsloth: Restored added_tokens_decoder metadata in outputs_adapter1_gemma4_e2b/checkpoint-1250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_adapter1_gemma4_e2b/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_adapter1_gemma4_e2b/checkpoint-1750/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_adapter1_gemma4_e2b/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_adapter1_gemma4_e2b/checkpoint-2091/tokenizer_config.json.
Could not locate the best model at outputs_adapter1_gemma4_e2b/checkpoint-250/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/adapter_1_gemma4_e2b_cuad/tokenizer_config.json.


Adapter saved to /kaggle/working/adapter_1_gemma4_e2b_cuad


## Evaluation Methodology and Metrics

This section evaluates the fine-tuned Gemma 4 E2B model on the held-out CUAD test set. The evaluation is designed around the actual objective of Adapter 1: **given a CUAD category and a relevant contract section, extract the exact supporting text or correctly return `NO_ANSWER`.**

Before comparison, the predicted and ground-truth text is normalized by converting text to lowercase, removing common articles and punctuation, and standardizing whitespace. This prevents superficial formatting differences from disproportionately affecting the evaluation.

### 1. Precision

**Precision** measures how much of the text generated by the model is actually part of the ground-truth answer.

A high precision value indicates that the model avoids returning unnecessary surrounding contract text and focuses on the relevant clause.

This is particularly important for CUAD because the task is **span extraction**, not general summarization. Returning a large amount of related text may appear relevant but is still an incorrect extraction.

### 2. Recall

**Recall** measures how much of the ground-truth CUAD answer was successfully recovered by the model.

High recall indicates that the model is not cutting off important portions of the contractual clause.

Recall is important because a prediction containing only part of the required clause may be insufficient for downstream legal analysis, even when the extracted portion is technically relevant.

### 3. F1 Score

**F1** combines precision and recall into a single score using their harmonic mean.

It is used as the primary overall extraction metric because CUAD requires a balance between:

- avoiding unnecessary text, and
- recovering the complete relevant clause.

A model that achieves high recall by returning entire paragraphs, for example, should not receive the same credit as a model that extracts the relevant span precisely.

### 4. Exact Match (EM)

**Exact Match** measures whether the normalized model prediction is exactly identical to the normalized CUAD ground-truth answer.

This is a stricter metric than F1.

Exact Match is especially valuable for this project because the system is intended to provide **evidence from the original contract**, rather than merely generate text that is semantically similar to the correct clause.

A high EM score therefore indicates that the model is consistently reproducing the expected contractual span.

### 5. No-Answer Accuracy

CUAD contains examples where the requested category is not present in the provided section. These cases are important because the model must distinguish:

`relevant clause exists`  
from  
`relevant clause is absent`

For these examples, the model is expected to return:

`NO_ANSWER`

**No-Answer Accuracy** measures how frequently the model correctly identifies these negative cases.

This metric is important for detecting models that aggressively extract text even when supporting evidence does not exist. Such behavior would be especially problematic in a legal system because it could introduce unsupported clauses into later analysis.

### 6. Macro-F1

The CUAD dataset contains multiple legal clause categories, and performance can vary significantly between categories.

**Macro-F1** is calculated by computing the F1 score independently for each category and then averaging those category-level scores.

This prevents categories with many examples from dominating the overall result.

Macro-F1 is therefore used to determine whether the model has learned a broadly useful CUAD extraction capability rather than performing well only on a small subset of familiar clause types.

### 7. Evidence Grounding Rate

**Evidence Grounding Rate** checks whether the normalized text generated by the model can be found within the original contract context.

The prediction is compared against the **full original contract section**, rather than only the answer-aware chunk supplied during inference.

This metric provides an additional safeguard against hallucination by checking whether the generated evidence is actually supported by the source document.

A high grounding rate indicates that the model's extracted text remains anchored to the original contractual evidence.

### Why These Metrics Are Used Together

No single metric fully describes legal clause extraction performance.

The evaluation therefore combines several complementary perspectives:

`Precision` → How much unnecessary text did the model avoid?

`Recall` → Did the model capture the complete relevant clause?

`F1` → Does the model balance precision and recall?

`Exact Match` → Did it reproduce the expected CUAD span exactly?

`No-Answer Accuracy` → Can it correctly recognize missing clauses?

`Macro-F1` → Does performance remain consistent across CUAD categories?

`Evidence Grounding Rate` → Is the generated evidence actually present in the source contract?

Together, these metrics provide a more complete assessment of whether the adapter has learned reliable, precise, and source-grounded CUAD extraction behavior.

In [28]:
# ============================================================
# Evaluation — Cell 1: Metrics & Text Normalization
# ============================================================

FastLanguageModel.for_inference(model)

NO_ANSWER = "NO_ANSWER"


# ------------------------------------------------------------
# Text normalization
# ------------------------------------------------------------

def normalize_text(s):
    """
    Normalize text for CUAD-style comparison.

    Used for:
        - Precision
        - Recall
        - F1
        - Exact Match
        - Grounding checks
    """

    if s is None:
        return ""

    # Make sure we are always working with a string
    s = str(s).strip()

    if not s:
        return ""

    # Lowercase
    s = s.lower()

    # Remove articles
    s = re.sub(r'\b(a|an|the)\b', ' ', s)

    # Remove punctuation
    exclude = set(string.punctuation)
    s = ''.join(
        ch for ch in s
        if ch not in exclude
    )

    # Normalize whitespace
    s = ' '.join(s.split())

    return s


# ------------------------------------------------------------
# Tokenization
# ------------------------------------------------------------

def get_tokens(s):
    normalized = normalize_text(s)

    if not normalized:
        return []

    return normalized.split()


# ------------------------------------------------------------
# Token-level metrics
# ------------------------------------------------------------

def calculate_metrics(pred, gt):
    """
    Calculate token-level:

        Precision
        Recall
        F1
        Exact Match

    Returns:
        precision, recall, f1, exact_match
    """

    pred_tokens = get_tokens(pred)
    gt_tokens = get_tokens(gt)

    # --------------------------------------------------------
    # Special case: both are no-answer
    # --------------------------------------------------------

    if (
        normalize_text(pred) == normalize_text(NO_ANSWER)
        and
        normalize_text(gt) == normalize_text(NO_ANSWER)
    ):
        return 1.0, 1.0, 1.0, 1


    # --------------------------------------------------------
    # Empty ground truth
    # --------------------------------------------------------

    if len(gt_tokens) == 0:

        # Empty prediction = correct
        if len(pred_tokens) == 0:
            return 1.0, 1.0, 1.0, 1

        # Model predicted something when there is no answer
        return 0.0, 0.0, 0.0, 0


    # --------------------------------------------------------
    # Ground truth exists but prediction is empty
    # --------------------------------------------------------

    if len(pred_tokens) == 0:
        return 0.0, 0.0, 0.0, 0


    # --------------------------------------------------------
    # Token overlap
    # --------------------------------------------------------

    common = (
        collections.Counter(pred_tokens)
        &
        collections.Counter(gt_tokens)
    )

    num_same = sum(common.values())


    # --------------------------------------------------------
    # Precision / Recall
    # --------------------------------------------------------

    precision = num_same / len(pred_tokens)

    recall = num_same / len(gt_tokens)


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    if precision + recall == 0:

        f1 = 0.0

    else:

        f1 = (
            2 * precision * recall
        ) / (
            precision + recall
        )


    # --------------------------------------------------------
    # Exact Match
    # --------------------------------------------------------

    em = int(
        normalize_text(pred)
        ==
        normalize_text(gt)
    )


    return precision, recall, f1, em

In [30]:
# ============================================================
# CUAD Evaluation
# ============================================================

print("\nStarting CUAD evaluation...")

results = {
    "precision": [],
    "recall": [],
    "f1": [],
    "em": [],
    "no_answer_acc": [],
    "grounding_rate": [],
    "categories": {}
}


# ------------------------------------------------------------
# Evaluation control
# ------------------------------------------------------------

# Quick evaluation:
# EVAL_LIMIT = 50

# Official evaluation:
EVAL_LIMIT = None


if EVAL_LIMIT is None:

    evaluation_dataset = test_dataset_raw

else:

    evaluation_dataset = test_dataset_raw.select(
        range(
            min(
                EVAL_LIMIT,
                len(test_dataset_raw)
            )
        )
    )


print(
    f"Evaluating {len(evaluation_dataset)} examples..."
)


# ------------------------------------------------------------
# Evaluation Loop
# ------------------------------------------------------------

for i, example in enumerate(evaluation_dataset):

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    has_answer = len(example["answers"]) > 0

    if has_answer:

        gt = example["answers"][0]["text"]

    else:

        gt = NO_ANSWER


    category = example["question"]


    # --------------------------------------------------------
    # Build the SAME answer-aware chunk used during training
    # --------------------------------------------------------

    chunk = make_answer_aware_chunk(example)

    user_message = render_user_message(
        category,
        chunk["chunk_context"]
    )


    # --------------------------------------------------------
    # Build Gemma chat prompt
    # --------------------------------------------------------

    messages = [
        {
            "role": "user",
            "content": user_message,
        }
    ]


    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    inputs = tokenizer(
        text=prompt,
        return_tensors="pt",
    ).to("cuda")


    input_length = inputs["input_ids"].shape[1]


    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )


    # --------------------------------------------------------
    # Decode ONLY newly generated tokens
    # --------------------------------------------------------

    generated_tokens = outputs[0][input_length:]

    pred = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()


    # --------------------------------------------------------
    # Basic metrics
    # --------------------------------------------------------

    p, r, f1, em = calculate_metrics(
        pred,
        gt
    )

    results["precision"].append(p)
    results["recall"].append(r)
    results["f1"].append(f1)
    results["em"].append(em)


    # --------------------------------------------------------
    # Category-level F1
    # --------------------------------------------------------

    if category not in results["categories"]:

        results["categories"][category] = []


    results["categories"][category].append(f1)


    # --------------------------------------------------------
    # No-answer accuracy
    # --------------------------------------------------------

    if not has_answer:

        normalized_pred = normalize_text(pred)

        is_no_answer = (
            normalized_pred
            == normalize_text(NO_ANSWER)
        )

        results["no_answer_acc"].append(
            int(is_no_answer)
        )


    # --------------------------------------------------------
    # Evidence grounding
    #
    # Check prediction against the FULL original contract,
    # not just the answer-aware chunk.
    # --------------------------------------------------------

    if has_answer:

        norm_context = normalize_text(
            example["context"]
        )

        norm_pred = normalize_text(pred)

        is_grounded = (
            bool(norm_pred)
            and norm_pred in norm_context
        )

        results["grounding_rate"].append(
            int(is_grounded)
        )


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 25 == 0:

        print(
            f"Processed "
            f"{i + 1}/{len(evaluation_dataset)}"
        )


Starting CUAD evaluation...
Evaluating 2091 examples...
Processed 25/2091
Processed 50/2091
Processed 75/2091
Processed 100/2091
Processed 125/2091
Processed 150/2091
Processed 175/2091
Processed 200/2091
Processed 225/2091
Processed 250/2091
Processed 275/2091
Processed 300/2091
Processed 325/2091
Processed 350/2091
Processed 375/2091
Processed 400/2091
Processed 425/2091
Processed 450/2091
Processed 475/2091
Processed 500/2091
Processed 525/2091
Processed 550/2091
Processed 575/2091
Processed 600/2091
Processed 625/2091
Processed 650/2091
Processed 675/2091
Processed 700/2091
Processed 725/2091
Processed 750/2091
Processed 775/2091
Processed 800/2091
Processed 825/2091
Processed 850/2091
Processed 875/2091
Processed 900/2091
Processed 925/2091
Processed 950/2091
Processed 975/2091
Processed 1000/2091
Processed 1025/2091
Processed 1050/2091
Processed 1075/2091
Processed 1100/2091
Processed 1125/2091
Processed 1150/2091
Processed 1175/2091
Processed 1200/2091
Processed 1225/2091
Proce

# 5. PRINT FINAL METRICS REPORT

In [31]:
# ============================================================
# Final Evaluation Results
# ============================================================

print("\n")
print("=" * 60)
print("CUAD EVALUATION RESULTS")
print("=" * 60)


# ------------------------------------------------------------
# Overall Metrics
# ------------------------------------------------------------

def safe_mean(values):
    """
    Return the mean of a list.
    Returns 0.0 if the list is empty.
    """
    if not values:
        return 0.0

    return sum(values) / len(values)


precision = safe_mean(
    results["precision"]
)

recall = safe_mean(
    results["recall"]
)

f1 = safe_mean(
    results["f1"]
)

em = safe_mean(
    results["em"]
)


print(f"Precision:              {precision:.4f}")
print(f"Recall:                 {recall:.4f}")
print(f"F1:                     {f1:.4f}")
print(f"Exact Match:            {em:.4f}")


# ------------------------------------------------------------
# No-Answer Accuracy
# ------------------------------------------------------------

if results["no_answer_acc"]:

    no_answer_acc = safe_mean(
        results["no_answer_acc"]
    )

    print(
        f"No-Answer Accuracy:     {no_answer_acc:.4f}"
    )

else:

    print(
        "No-Answer Accuracy:     N/A"
    )


# ------------------------------------------------------------
# Evidence Grounding Rate
# ------------------------------------------------------------

if results["grounding_rate"]:

    grounding_rate = safe_mean(
        results["grounding_rate"]
    )

    print(
        f"Evidence Grounding Rate: {grounding_rate:.4f}"
    )

else:

    print(
        "Evidence Grounding Rate: N/A"
    )


# ------------------------------------------------------------
# Macro-F1
# ------------------------------------------------------------

category_f1 = {}

for category, scores in results["categories"].items():

    if scores:

        category_f1[category] = safe_mean(
            scores
        )


if category_f1:

    macro_f1 = safe_mean(
        list(category_f1.values())
    )

    print(
        f"Macro-F1:               {macro_f1:.4f}"
    )

else:

    print(
        "Macro-F1:               N/A"
    )


# ------------------------------------------------------------
# Evaluation Summary
# ------------------------------------------------------------

print("=" * 60)

print(
    f"Examples evaluated:      "
    f"{len(results['f1'])}"
)

print(
    f"Categories evaluated:   "
    f"{len(category_f1)}"
)

print("=" * 60)



CUAD EVALUATION RESULTS
Precision:              0.8116
Recall:                 0.8096
F1:                     0.8024
Exact Match:            0.7484
No-Answer Accuracy:     0.9787
Evidence Grounding Rate: 0.5854
Macro-F1:               0.8024
Examples evaluated:      2091
Categories evaluated:   41
